## **Step 1: Import Libraries**

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

import joblib

## **Step 2: Load Dataset**

In [ ]:
df = pd.read_csv("soc_analyst_dataset_200k.csv")

print(df.shape)
df.head()

(200000, 14)


,alert_id,alert_source,alert_type,severity_raw,failed_login_count,geo_anomaly,known_ioc_match,malware_detected,asset_criticality,user_risk_score,overall_risk_score,threat_label,priority_label,response_action
0,ALT-000000,DLP,Phishing,9,2,0,0,0,Low,13,52.25,Benign,P4,Close Alert
1,ALT-000001,IDS,Data Exfiltration,2,1,0,0,0,Medium,83,32.75,Benign,P4,Close Alert
2,ALT-000002,Email Gateway,Brute Force,1,4,1,0,0,Low,70,45.50,Benign,P4,Close Alert
3,ALT-000003,IPS,Data Exfiltration,1,5,0,0,0,Low,42,25.50,Benign,P4,Close Alert
4,ALT-000004,DLP,Lateral Movement,3,3,0,0,0,High,23,26.75,Benign,P4,Close Alert


## **Step 3: Target Variable**

In [ ]:
target = "threat_label"

Convert labels:

In [ ]:
df[target] = df[target].map({
    "Benign":0,
    "Malicious":1
})

## **Step 4: Drop Non-Predictive Columns**

In [ ]:
drop_cols = [
    "alert_id",
    "threat_label",
    "priority_label",
    "response_action"
]

X = df.drop(columns=drop_cols)
y = df[target]

## **Step 5: Separate Feature Types**




In [ ]:
categorical_cols = X.select_dtypes(
    include=["object"]
).columns.tolist()

numeric_cols = X.select_dtypes(
    exclude=["object"]
).columns.tolist()

print("Categorical:", len(categorical_cols))
print("Numeric:", len(numeric_cols))

Categorical: 3
Numeric: 7


## **Step 6: Preprocessing Pipeline**

Numeric

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)

Categorical

In [ ]:
categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

Combine

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_cols
        ),
        (
            "cat",
            categorical_transformer,
            categorical_cols
        )
    ]
)

## **Step 7: Train/Test Split**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## **Step 8: Model Selection**

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

## **Step 9: Build Full Pipeline**

In [ ]:
model_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", rf_model)
    ]
)

## **Step 10: Train Model**

In [ ]:
model_pipeline.fit(
    X_train,
    y_train
)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['severity_raw',
                                                   'failed_login_count',
                                                   'geo_anomaly',
                                                   'known_ioc_match',
                                                   'malware_detected',
                                                   'user_risk_score',
                                                   'overall_risk_score']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['alert_source', 'alert_type',
                                                   'asset_criticality'])])),
                ('classifier',
                 RandomForestClassifier(max_depth=15, n_estimators=200,
                                        n_jobs=-1, random_state=42))])

## **Step 11: Evaluate**

In [ ]:
y_pred = model_pipeline.predict(X_test)

print(
    "Accuracy:",
    accuracy_score(
        y_test,
        y_pred
    )
)

print(
    classification_report(
        y_test,
        y_pred
    )
)

print(
    confusion_matrix(
        y_test,
        y_pred
    )
)

Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     27293
           1       1.00      1.00      1.00     12707

    accuracy                           1.00     40000
   macro avg       1.00      1.00      1.00     40000
weighted avg       1.00      1.00      1.00     40000

[[27293     0]
 [    0 12707]]


## **Step 12: Save Model**

In [ ]:
joblib.dump(
    model_pipeline,
    "threat_detection_model.pkl"
)

['threat_detection_model.pkl']

Verify:

In [ ]:
import os

print(
    os.path.exists(
        "threat_detection_model.pkl"
    )
)

True


## **Test Prediction**

In [ ]:
sample = X_test.iloc[[22]]

prediction = model_pipeline.predict(sample)

print(prediction)

[1]


In [ ]:
print(X.columns.tolist())

['alert_source', 'alert_type', 'severity_raw', 'failed_login_count', 'geo_anomaly', 'known_ioc_match', 'malware_detected', 'asset_criticality', 'user_risk_score', 'overall_risk_score']
